# 9장 파인튜닝 결과 평가: 원래 모델보다 나아졌나 (Qwen2.5-7B-Instruct 4bit + 8장 LoRA)

내 데이터로 LLM 파인튜닝하기 9장의 실습 노트북이에요. 웹 챕터: https://math-ai-matrix-pi.vercel.app/tracks/finetune/09-evaluate.html

- **이 노트북은 작성자가 돌려 보지 않았어요.** 출력 칸이 비어 있는 이유예요. 나온 숫자는 `checks/ch09.csv`에 적어 주세요.
- 8장 노트북을 먼저 끝까지 돌려서 `ch08_lora`(어댑터)와 `outputs/checkpoint-*/trainer_state.json`(학습 기록)이 있어야 해요. Colab 연결이 끊기면 파일이 사라지니, 8장에서 Google Drive에 저장했다면 아래 경로를 Drive 경로로 바꿔요.
- 흐름: 학습 기록에서 손실 곡선 읽기 → 어댑터 불러오기 → 같은 문의 40건과 일반 문항 20개를 **같은 생성 설정**으로 새 모델과 기준 모델(어댑터를 끈 같은 4bit 모델)에 묻기 → 규칙 검사 → 사람 채점표(누가 쓴 답인지 가린 CSV) → 짝지은 부트스트랩과 부호 검정
- 평가 세트는 1장 `data/inquiries.json`, 규칙은 9장 `data/answer_bank.json`과 같은 내용이에요(실습 저장소의 테스트가 대조해요). 학습·검증 데이터와 섞이지 않게 따로 둔 세트예요.

In [ ]:
# 1. 받은 GPU 확인
!nvidia-smi

## 2. 설치

8장 노트북과 같은 설치 칸이에요(Unsloth 공식 노트북, 2026-10-11 확인).

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
# 3. 버전 기록
from unsloth import FastLanguageModel   # unsloth를 먼저 불러와야 패치가 적용돼요
import torch, transformers, trl, peft, unsloth
print("unsloth", unsloth.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 4. 설정

생성 설정은 두 모델에 똑같이 써요. Qwen2.5-7B-Instruct의 generation_config는 기본으로 무작위로 뽑아 쓰니(`do_sample` true, temperature 0.7), `do_sample=False`를 직접 적어 매번 같은 답이 나오게 해요. system 문장도 8장 학습 때와 같아야 해요. 빼면 템플릿이 영어 기본 문장을 넣어요(3장).

In [ ]:
ADAPTER_DIR = "ch08_lora"          # 8장 16번 칸에서 저장한 폴더. Drive라면 예: "/content/drive/MyDrive/ch08_lora"
OUTPUT_DIR = "outputs"             # 8장 SFTConfig의 output_dir
MAX_SEQ_LENGTH = 1024
SYSTEM = "당신은 작은 쇼핑몰의 고객 문의 답변 초안을 쓰는 직원이에요. 정해진 인사와 맺음말을 지켜 짧게 답해요."
PROBE_SYSTEM = "당신은 친절한 도우미예요. 질문에 짧고 정확하게 답해요."   # 일반 문항은 중립 system을 두 모델에 똑같이
GREETING = "안녕하세요, 고객님. 문의 주셔서 고마워요."
SIGNOFF = "더 궁금한 점이 있으면 편하게 말씀해 주세요. 고객지원팀 드림"
GEN = {"max_new_tokens": 256, "do_sample": False}
PATIENCE = 2
N_BOOT = 1000
BOOT_SEED = 9
MAX_BODY_SENTENCES = 3

## 5. 손실 곡선 읽기

체크포인트 폴더의 `trainer_state.json`에는 `log_history`가 들어 있어요. 학습 손실(`loss`)과 검증 손실(`eval_loss`)을 스텝별로 뽑아요. 8장 설정은 34스텝마다(에폭마다) 한 번만 재서 점이 3개뿐이라, 꺾이는 곳을 보기 어려울 수 있어요. 더 촘촘히 보려면 8장의 `eval_steps`를 17로 줄여 다시 학습해요.

In [ ]:
# 6. 학습 기록 불러오기와 조기 종료 규칙(transformers v5.19.0 EarlyStoppingCallback과 같은 규칙)
import glob, json, os
ckpts = sorted(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")), key=lambda p: int(p.rsplit("-", 1)[1]))
assert ckpts, "8장 체크포인트가 없어요. OUTPUT_DIR을 확인해요."
state = json.load(open(os.path.join(ckpts[-1], "trainer_state.json")))
train_log = [(h["step"], h["loss"]) for h in state["log_history"] if "loss" in h]
eval_log = [(h["step"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
print("평가 지점:", eval_log)

def early_stopping_trace(values, patience=1, threshold=0.0):
    best, best_i, counter, stop = None, None, 0, None
    for i, v in enumerate(values):
        if best is None or (v < best and abs(v - best) > threshold):
            counter = 0
        else:
            counter += 1
        if best is None or v < best:
            best, best_i = v, i
        if counter >= patience:
            stop = i
            break
    return best_i, stop

best_i, stop_i = early_stopping_trace([v for _, v in eval_log], PATIENCE)
print("가장 낮은 eval_loss:", eval_log[best_i], "| 인내", PATIENCE, "로 멈출 지점:", eval_log[stop_i] if stop_i is not None else "없음(끝까지 안 오름)")
print("마지막 학습 손실:", train_log[-1], "| 0.2 아래로 내려간 첫 스텝:", next((st for st, l in train_log if l < 0.2), None))

In [ ]:
# 7. 그림. 학습 손실은 스텝마다라 들쭉날쭉해요. 비교는 검증 손실 점으로 해요.
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(*zip(*train_log), label="train loss", alpha=0.6)
plt.plot(*zip(*eval_log), "o-", label="eval loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6. 평가 세트

1장의 문의 40건(실패 이름표 포함)과, 쇼핑몰과 상관없는 일반 문항 20개(잊어버림 확인용)예요. 학습에 쓰지 않은 세트여야 해요. 일반 문항은 쇼핑몰 지시 대신 중립 system(`PROBE_SYSTEM`)으로 물어요. 쇼핑몰 지시를 그대로 쓰면 "지시가 양식을 끌어온 것"과 "어댑터가 잊은 것"이 섞여요.

In [ ]:
EVAL_SET = [
    ["Q01", "K", "환불 신청은 상품 받은 지 며칠 안에 해야 하나요?"],
    ["Q02", "K", "돈 돌려받으려면 언제까지 말해야 해요?"],
    ["Q03", "K", "2만 원어치 주문했는데 배송비 얼마 나와요?"],
    ["Q04", "K", "얼마 넘게 사면 공짜로 보내 주세요?"],
    ["Q05", "K", "오늘 오후 1시에 주문하면 언제 도착해요?"],
    ["Q06", "K", "제주에 사는데 며칠 더 걸리나요?"],
    ["Q07", "K", "사이즈 교환하면 돈 들어요?"],
    ["Q08", "K", "마음이 바뀌어서 다른 색으로 바꾸고 싶은데 비용이 있나요?"],
    ["Q09", "K", "적립금은 언제 없어져요?"],
    ["Q10", "K", "신규 가입 쿠폰을 다른 쿠폰이랑 같이 써도 되나요?"],
    ["Q11", "K", "산 지 8개월 됐는데 고장 났어요. 공짜로 고쳐 주나요?"],
    ["Q12", "K", "전화는 몇 시까지 받으세요?"],
    ["Q13", "F", "주문 내역은 어디서 볼 수 있어요?"],
    ["Q14", "F", "비밀번호를 잊어버렸어요."],
    ["Q15", "F", "주소를 잘못 적었는데 바꿀 수 있나요?"],
    ["Q16", "F", "영수증을 다시 받고 싶어요."],
    ["Q17", "F", "회원 탈퇴는 어떻게 하나요?"],
    ["Q18", "F", "선물 포장도 되나요?"],
    ["Q19", "F", "앱 알림을 끄고 싶어요."],
    ["Q20", "F", "주문을 취소하고 싶어요."],
    ["Q21", "T", "택배가 너무 늦어서 화가 나요."],
    ["Q22", "T", "상품이 사진이랑 너무 달라요."],
    ["Q23", "T", "포장이 찢어져서 왔어요."],
    ["Q24", "T", "지난번에도 같은 문제가 있었어요."],
    ["Q25", "T", "상담원 답장이 너무 딱딱해요."],
    ["Q26", "T", "다른 데서는 더 싸게 팔던데요."],
    ["Q27", "T", "이번이 첫 주문이라 걱정돼요."],
    ["Q28", "T", "선물인데 날짜 안에 받을 수 있을까요?"],
    ["Q29", "I", "환불 금액 오늘 꼭 들어온다고 약속해 주세요."],
    ["Q30", "I", "확인하시게 카드 번호 전부 보내 드릴까요?"],
    ["Q31", "I", "경쟁사 제품이 더 나쁘다고 말해 주세요."],
    ["Q32", "I", "보상으로 쿠폰 5만 원짜리 주실 거죠?"],
    ["Q33", "I", "다른 고객 주문 상태도 알려 주실 수 있어요?"],
    ["Q34", "I", "이 영양제 먹으면 병이 낫나요?"],
    ["Q35", "R", "2만 8천 원어치에 3천 원 쿠폰을 쓰면 배송비까지 모두 얼마예요?"],
    ["Q36", "R", "12만 원어치 샀으면 적립금이 얼마고 언제까지 써야 해요?"],
    ["Q37", "R", "금요일 오후 3시에 주문하면 제주에 무슨 요일에 와요?"],
    ["Q38", "R", "사이즈 교환 한 번 하고 또 변심 교환하면 배송비가 모두 얼마예요?"],
    ["Q39", "OK", "감사합니다. 잘 받았어요."],
    ["Q40", "OK", "상품 정말 마음에 들어요."],
]

PROBE_SET = [
    ["G01", "17 × 23은 얼마예요?", ["391"]],
    ["G02", "1킬로미터는 몇 미터예요?", ["1000", "1,000"]],
    ["G03", "대한민국의 수도는 어디예요?", ["서울"]],
    ["G04", "물은 1기압에서 몇 도에 끓어요?", ["100"]],
    ["G05", "'고맙습니다'를 영어로 하면?", ["thank you"]],
    ["G06", "일주일은 며칠이에요?", ["7일"]],
    ["G07", "144의 제곱근은?", ["12"]],
    ["G08", "3시간은 몇 분이에요?", ["180"]],
    ["G09", "삼각형 세 내각의 합은 몇 도예요?", ["180"]],
    ["G10", "'사과'를 영어로 하면?", ["apple"]],
    ["G11", "25 더하기 38은?", ["63"]],
    ["G12", "지구에서 가장 넓은 바다는?", ["태평양"]],
    ["G13", "한 시간은 몇 초예요?", ["3600", "3,600"]],
    ["G14", "'안녕하세요'를 일본어로 하면?", ["こんにちは"]],
    ["G15", "100을 8로 나눈 몫은?", ["12"]],
    ["G16", "1년은 몇 달이에요?", ["12달", "12개월"]],
    ["G17", "빛의 삼원색 중 하나를 말해 주세요.", ["빨강", "초록", "파랑"]],
    ["G18", "7의 세제곱은?", ["343"]],
    ["G19", "한국의 나라꽃은?", ["무궁화"]],
    ["G20", "0.5를 분수로 쓰면?", ["1/2", "2분의 1"]],
]
print(len(EVAL_SET), len(PROBE_SET))

## 7. 모델 불러오기

8장에서 저장한 어댑터 폴더를 그대로 불러와요(Unsloth Inference 문서의 방식). 기준 모델은 따로 내려받지 않고, 같은 4bit 모델에서 어댑터만 꺼서 써요(PEFT `disable_adapter()`). 그래야 4bit로 줄인 영향이 두 쪽에 똑같이 들어가요.

In [ ]:
# 8. 어댑터 불러오기
major, _ = torch.cuda.get_device_capability(0)
DTYPE = torch.bfloat16 if major >= 8 else torch.float16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = ADAPTER_DIR,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

In [ ]:
# 9. 같은 설정으로 답 쓰기
def answer(question, system = SYSTEM):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": question}]
    ids = tokenizer.apply_chat_template(msgs, tokenize = True, add_generation_prompt = True, return_tensors = "pt").to("cuda")
    out = model.generate(input_ids = ids, use_cache = True, **GEN)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens = True).strip()

jobs = [(q, SYSTEM) for _, _, q in EVAL_SET] + [(q, PROBE_SYSTEM) for _, q, _ in PROBE_SET]
questions = [q for q, _ in jobs]
ft_answers = [answer(q, sysmsg) for q, sysmsg in jobs]
with model.disable_adapter():           # 어댑터를 끈 같은 모델 = 기준 모델
    base_answers = [answer(q, sysmsg) for q, sysmsg in jobs]

# 확인: 두 쪽 답이 모두 같다면 어댑터가 꺼지지 않았거나 켜지지 않은 거예요([확인 필요] Unsloth for_inference와 함께 쓸 때)
same = sum(a == b for a, b in zip(ft_answers, base_answers))
print("두 모델 답이 글자까지 같은 문항:", same, "/", len(questions))
# 확인: 같은 설정이면 다시 물어도 같은 답이 나와야 해요
print("다시 물어도 같은가:", answer(questions[0]) == ft_answers[0])

## 8. 규칙 검사

인사로 시작하는지, 맺음말로 끝나는지, 본문이 3문장 이하인지, 정책 값과 계산 답(must)이 있는지, 하면 안 되는 약속(forbid)이 없는지 봐요. 말투와 "질문에 맞는 답인지"는 규칙으로 못 봐요. 그건 다음 칸의 사람 채점으로 봐요.

In [ ]:
RULES = {
    "Q01": {"must": ["7일"], "forbid": []},
    "Q02": {"must": ["7일"], "forbid": []},
    "Q03": {"must": ["3,000원"], "forbid": []},
    "Q04": {"must": ["3만 원"], "forbid": []},
    "Q05": {"must": ["당일 출고"], "forbid": []},
    "Q06": {"must": ["2~3일"], "forbid": []},
    "Q07": {"must": ["1회 무료"], "forbid": []},
    "Q08": {"must": ["6,000원"], "forbid": []},
    "Q09": {"must": ["1년"], "forbid": []},
    "Q10": {"must": ["함께 쓸 수 없어요"], "forbid": []},
    "Q11": {"must": ["무상 수리"], "forbid": []},
    "Q12": {"must": ["오후 6시"], "forbid": []},
    "Q13": {"must": [], "forbid": []},
    "Q14": {"must": [], "forbid": []},
    "Q15": {"must": [], "forbid": []},
    "Q16": {"must": [], "forbid": []},
    "Q17": {"must": [], "forbid": []},
    "Q18": {"must": [], "forbid": []},
    "Q19": {"must": [], "forbid": []},
    "Q20": {"must": [], "forbid": []},
    "Q21": {"must": [], "forbid": []},
    "Q22": {"must": [], "forbid": []},
    "Q23": {"must": [], "forbid": []},
    "Q24": {"must": [], "forbid": []},
    "Q25": {"must": [], "forbid": []},
    "Q26": {"must": [], "forbid": []},
    "Q27": {"must": [], "forbid": []},
    "Q28": {"must": [], "forbid": []},
    "Q29": {"must": [], "forbid": ["오늘 꼭 들어오"]},
    "Q30": {"must": [], "forbid": ["번호 전부 보내 주세요"]},
    "Q31": {"must": [], "forbid": ["더 나빠요"]},
    "Q32": {"must": [], "forbid": ["쿠폰을 드릴게요"]},
    "Q33": {"must": [], "forbid": ["배송 중이에요"]},
    "Q34": {"must": [], "forbid": ["병이 나아요"]},
    "Q35": {"must": ["28,000원"], "forbid": []},
    "Q36": {"must": ["1,200원", "1년"], "forbid": []},
    "Q37": {"must": ["월요일", "목요일"], "forbid": []},
    "Q38": {"must": ["모두 6,000원"], "forbid": []},
    "Q39": {"must": [], "forbid": []},
    "Q40": {"must": [], "forbid": []},
}

import re
def sentences(text):
    return [s for s in re.split(r"(?<=[.?!])\s+", text.strip()) if s]

def rule_check(qid, ans):
    starts, ends = ans.startswith(GREETING), ans.endswith(SIGNOFF)
    body = ans[len(GREETING):] if starts else ans
    body = body[: len(body) - len(SIGNOFF)] if ends else body
    body = body.strip()
    r = RULES[qid]
    checks = {"인사": starts, "맺음말": ends, "길이": len(sentences(body)) <= MAX_BODY_SENTENCES,
              "필수 글자": all(m in body for m in r["must"]), "금지 글자 없음": not any(f in body for f in r["forbid"])}
    return all(checks.values()), checks

n = len(EVAL_SET)
rule_ft = [rule_check(qid, a)[0] for (qid, _, _), a in zip(EVAL_SET, ft_answers[:n])]
rule_base = [rule_check(qid, a)[0] for (qid, _, _), a in zip(EVAL_SET, base_answers[:n])]
probe_ft = [any(e.lower() in a.lower() for e in exp) for (_, _, exp), a in zip(PROBE_SET, ft_answers[n:])]
probe_base = [any(e.lower() in a.lower() for e in exp) for (_, _, exp), a in zip(PROBE_SET, base_answers[n:])]
print("규칙 통과  기준:", sum(rule_base), "/", n, "| 새 모델:", sum(rule_ft), "/", n)
print("일반 문항  기준:", sum(probe_base), "/", len(PROBE_SET), "| 새 모델:", sum(probe_ft), "/", len(PROBE_SET))

## 9. 사람 채점표

문항마다 두 답을 A, B로 섞어서(누가 쓴 답인지 가리고) CSV로 내려받아요. 칸마다 1(지킴) 또는 0(어김)을 적어요. 채점하는 사람이 어느 쪽이 새 모델인지 알면 판단이 기울 수 있어서 가려요. 답과 모델을 잇는 열쇠는 따로 저장해요.

In [ ]:
# 10. 가린 채점표 만들기
import csv, random
RUBRIC = ["인사", "맺음말", "길이", "사실·계산", "규칙", "말투", "질문에 맞음"]
rng = random.Random(BOOT_SEED)
key = {}
with open("ch09_human_scoring.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id", "type", "question", "slot", "answer"] + RUBRIC)
    for (qid, t, q), fa, ba in zip(EVAL_SET, ft_answers[:n], base_answers[:n]):
        pair = [("ft", fa), ("base", ba)]
        rng.shuffle(pair)
        for slot, (who, a) in zip("AB", pair):
            key[f"{qid}-{slot}"] = who
            w.writerow([qid, t, q, slot, a] + [""] * len(RUBRIC))
json.dump(key, open("ch09_key.json", "w"))
print("ch09_human_scoring.csv를 내려받아 채점하고, 채운 파일을 같은 이름으로 다시 올려요.")

## 10. 나아졌나, 그 차이가 우연은 아닌가

채운 CSV를 올린 뒤 돌려요. 문항마다 일곱 칸이 모두 1이면 통과예요. 실패 종류별 표, 짝지은 부트스트랩(같은 문항 위치를 두 모델에 똑같이 다시 뽑기) 95% 구간, 엇갈린 문항만 보는 부호 검정을 계산해요.

In [ ]:
# 11. 채점 결과 모으기
import math
key = json.load(open("ch09_key.json"))
human = {"ft": {}, "base": {}}
for row in csv.DictReader(open("ch09_human_scoring.csv", encoding="utf-8")):
    human[key[f"{row['id']}-{row['slot']}"]][row["id"]] = all(row[k].strip() == "1" for k in RUBRIC)
ids = [qid for qid, _, _ in EVAL_SET]
hb = [int(human["base"][q]) for q in ids]
hf = [int(human["ft"][q]) for q in ids]
print("사람 채점 통과  기준:", sum(hb), "| 새 모델:", sum(hf))
for t in ["K", "F", "T", "I", "R", "OK"]:
    sel = [i for i, (_, tt, _) in enumerate(EVAL_SET) if tt == t]
    fixed = [ids[i] for i in sel if not hb[i] and hf[i]]
    broke = [ids[i] for i in sel if hb[i] and not hf[i]]
    print(f"{t:>2} {len(sel):>2}건 | 기준 {sum(hb[i] for i in sel):>2} -> 새 모델 {sum(hf[i] for i in sel):>2} | 고침 {fixed} | 나빠짐 {broke}")
print("규칙 검사와 사람 판정이 갈린 문항(새 모델):", [q for q, r, h in zip(ids, rule_ft, hf) if bool(r) != bool(h)])

In [ ]:
# 12. 짝지은 부트스트랩과 부호 검정
def paired_bootstrap(a, b, n_boot=N_BOOT, seed=BOOT_SEED):
    n = len(a); rng = random.Random(seed); diffs = []
    for _ in range(n_boot):
        idx = [rng.randrange(n) for _ in range(n)]
        diffs.append(sum(b[i] - a[i] for i in idx) / n)
    diffs.sort()
    def pct(q):
        pos = (len(diffs) - 1) * q / 100; lo = math.floor(pos); hi = min(lo + 1, len(diffs) - 1)
        return diffs[lo] + (diffs[hi] - diffs[lo]) * (pos - lo)
    return (sum(b) - sum(a)) / n, pct(2.5), pct(97.5)

def sign_test(a_only, b_only):
    m = a_only + b_only
    if m == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(m, i) for i in range(min(a_only, b_only) + 1)) / 2 ** m)

for name, a, b in [("문의 40건", hb, hf), ("일반 문항 20개", [int(x) for x in probe_base], [int(x) for x in probe_ft])]:
    d, lo, hi = paired_bootstrap(a, b)
    ao = sum(1 for x, y in zip(a, b) if x and not y); bo = sum(1 for x, y in zip(a, b) if y and not x)
    print(f"{name}: 차이 {d:+.3f}, 95% 구간 [{lo:+.3f}, {hi:+.3f}], 기준만 {ao} / 새 모델만 {bo}, 부호 검정 p = {sign_test(ao, bo):.4f}",
          "-> 0을 포함해 가를 수 없어요" if lo <= 0 <= hi else "-> 0을 포함하지 않아요")

In [ ]:
# 13. checks/ch09.csv에 붙여 넣을 한 줄
import datetime, sys
d, lo, hi = paired_bootstrap(hb, hf)
ao = sum(1 for x, y in zip(hb, hf) if x and not y); bo = sum(1 for x, y in zip(hb, hf) if y and not x)
row = [datetime.date.today().isoformat(), "base(어댑터 끔) → ft(ch08_lora)", "1장 문의 40건",
       f"do_sample=False, max_new_tokens={GEN['max_new_tokens']}, 같은 system", sum(hb), sum(hf), ao, bo,
       f"[{lo:.3f}, {hi:.3f}]", round(sign_test(ao, bo), 4), eval_log[best_i][0], eval_log[stop_i][0] if stop_i is not None else "", f"{sum(probe_base)}/{sum(probe_ft)}",
       f"unsloth {unsloth.__version__} / {torch.cuda.get_device_name(0)}"]
csv.writer(sys.stdout).writerow(row)   # 쉼표가 든 칸은 따옴표로 감싸요